# KoELECTRA Core Audit Experiments

Legal-conclusion ablation and outcome-cue insertion experiments.

Set the private project root before execution:

```python
import os
os.environ['KLJPAUDIT_ROOT'] = '/path/to/K-LJPAUDIT'
```

Do not publish raw judicial facts, transformed text, per-case predictions, case identifiers, review files, or model checkpoints.

## Execution block 1

In [ ]:
import os
# ============================================================
# KoELECTRA LEGAL-CONCLUSION ABLATION GENERALIZATION
# Conditions: original / marker_removed / sentence_removed
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import gc
import json
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(os.environ.get("KLJPAUDIT_ROOT", "."))

EXPERIMENT_NAME = (
    "koelectra_legal_conclusion_ablation_v1"
)

RESULT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "audit_experiments"
    / EXPERIMENT_NAME
)

PRIVATE_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / EXPERIMENT_NAME
)

PREDICTION_DIR = RESULT_ROOT / "predictions"
METRICS_DIR = RESULT_ROOT / "metrics"
BOOTSTRAP_DIR = RESULT_ROOT / "bootstrap"

for path in [
    RESULT_ROOT,
    PRIVATE_DATA_DIR,
    PREDICTION_DIR,
    METRICS_DIR,
    BOOTSTRAP_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "monologg/koelectra-base-v3-discriminator"

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "checkpoints"
    / "koelectra_base_v3_multitask_seed42"
    / "best_model.pt"
)

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

TARGETS = [
    "fine_lv",
    "imprisonment_with_labor_lv",
    "imprisonment_without_labor_lv",
]

NUM_LABELS = {
    "fine_lv": 5,
    "imprisonment_with_labor_lv": 6,
    "imprisonment_without_labor_lv": 5,
}

CONDITIONS = [
    "original",
    "marker_removed",
    "sentence_removed",
]

MARKER = "이로써"
REQUIRED_TERM = "피고인"

MAX_LENGTH = 512
BATCH_SIZE = 32
BOOTSTRAP_ITERATIONS = 1000
SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 72)
print("KoELECTRA LEGAL-CONCLUSION ABLATION")
print("=" * 72)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Checkpoint:", CHECKPOINT_PATH)
print("Result root:", RESULT_ROOT)


# ------------------------------------------------------------
# 2. Reproducibility
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# 3. Exact legal-conclusion transformations
# ------------------------------------------------------------

def normalize_space(text):
    text = re.sub(
        r"[ \t]+",
        " ",
        str(text),
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()


def sentence_spans(text):
    pattern = re.compile(
        r"[^.!?\n。]+(?:[.!?。]+|\n+|$)",
        re.MULTILINE,
    )

    return [
        {
            "start": match.start(),
            "end": match.end(),
            "sentence": match.group(0),
        }
        for match in pattern.finditer(str(text))
        if match.group(0).strip()
    ]


def locate_conclusion_sentence(
    text,
    marker=MARKER,
    required_term=REQUIRED_TERM,
):
    matches = [
        span
        for span in sentence_spans(text)
        if marker in span["sentence"]
        and required_term in span["sentence"]
    ]

    return matches[-1] if matches else None


def remove_span(text, start, end):
    left = text[:start].rstrip()
    right = text[end:].lstrip()

    separator = " " if left and right else ""

    return normalize_space(
        left + separator + right
    )


def build_ablation_variants(text):
    original = normalize_space(text)

    span = locate_conclusion_sentence(original)

    if span is None:
        return {
            "matched": False,
            "original": original,
            "marker_removed": original,
            "sentence_removed": original,
            "selected_sentence": "",
        }

    selected_sentence = span["sentence"]

    marker_removed_sentence = (
        selected_sentence.replace(MARKER, "")
    )

    marker_removed = normalize_space(
        original[:span["start"]]
        + marker_removed_sentence
        + original[span["end"]:]
    )

    sentence_removed = remove_span(
        original,
        span["start"],
        span["end"],
    )

    return {
        "matched": True,
        "original": original,
        "marker_removed": marker_removed,
        "sentence_removed": sentence_removed,
        "selected_sentence": selected_sentence.strip(),
    }


# ------------------------------------------------------------
# 4. Load verified test split
# ------------------------------------------------------------

dataset = load_dataset(
    "lbox/lbox_open",
    "ljp_criminal",
)

test_raw = dataset["test"]

assert len(test_raw) == 1050

records = []

for row in test_raw:
    variants = build_ablation_variants(
        row["facts"]
    )

    record = {
        "case_uid": str(row["id"]),
        "matched": variants["matched"],
        "text_original": variants["original"],
        "text_marker_removed": (
            variants["marker_removed"]
        ),
        "text_sentence_removed": (
            variants["sentence_removed"]
        ),
        "selected_conclusion_sentence": (
            variants["selected_sentence"]
        ),
    }

    for target in TARGETS:
        record[target] = int(
            row["label"][target]
        )

    records.append(record)

full_df = pd.DataFrame(records)

assert len(full_df) == 1050
assert full_df["case_uid"].nunique() == 1050

matched_df = (
    full_df.loc[full_df["matched"]]
    .copy()
    .reset_index(drop=True)
)

print("\nTransformation coverage:")
print(" Total test cases:", len(full_df))
print(" Matched cases:", len(matched_df))
print(
    " Matched rate:",
    f"{len(matched_df) / len(full_df):.2%}",
)

assert len(matched_df) == 351, (
    f"Expected 351 matched cases, "
    f"found {len(matched_df)}"
)

assert (
    matched_df["text_original"]
    != matched_df["text_marker_removed"]
).all()

assert (
    matched_df["text_original"]
    != matched_df["text_sentence_removed"]
).all()

private_data_path = (
    PRIVATE_DATA_DIR
    / "koelectra_legal_conclusion_ablation_private.parquet"
)

matched_df.to_parquet(
    private_data_path,
    index=False,
)


# ------------------------------------------------------------
# 5. Tokenizer and model
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)


class KoElectraMultiTaskClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME
        )

        hidden_size = (
            self.encoder.config.hidden_size
        )

        self.dropout = nn.Dropout(0.10)

        self.classifiers = nn.ModuleDict({
            target: nn.Linear(
                hidden_size,
                NUM_LABELS[target],
            )
            for target in TARGETS
        })

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
    ):
        inputs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        if token_type_ids is not None:
            inputs["token_type_ids"] = (
                token_type_ids
            )

        outputs = self.encoder(**inputs)

        cls_vector = (
            outputs.last_hidden_state[:, 0, :]
        )

        cls_vector = self.dropout(cls_vector)

        return {
            target: classifier(cls_vector)
            for target, classifier
            in self.classifiers.items()
        }


model = KoElectraMultiTaskClassifier()

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

state_dict = checkpoint.get(
    "model_state_dict",
    checkpoint,
)

if all(
    key.startswith("module.")
    for key in state_dict.keys()
):
    state_dict = {
        key[len("module."):]: value
        for key, value in state_dict.items()
    }

load_result = model.load_state_dict(
    state_dict,
    strict=True,
)

print("\nCheckpoint loading:")
print(" Missing keys:", load_result.missing_keys)
print(
    " Unexpected keys:",
    load_result.unexpected_keys,
)

assert not load_result.missing_keys
assert not load_result.unexpected_keys

model = model.to(DEVICE)
model.eval()


# ------------------------------------------------------------
# 6. Inference dataset
# ------------------------------------------------------------

class TextInferenceDataset(Dataset):
    def __init__(
        self,
        dataframe,
        text_column,
    ):
        self.dataframe = dataframe.reset_index(
            drop=True
        )
        self.text_column = text_column

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        encoded = tokenizer(
            str(row[self.text_column]),
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
            return_attention_mask=True,
        )

        output = {
            "case_uid": str(row["case_uid"]),
            "input_ids": encoded["input_ids"],
            "attention_mask": (
                encoded["attention_mask"]
            ),
        }

        if "token_type_ids" in encoded:
            output["token_type_ids"] = (
                encoded["token_type_ids"]
            )

        for target in TARGETS:
            output[target] = int(row[target])

        return output


def collate_batch(batch):
    features = []

    for row in batch:
        feature = {
            "input_ids": row["input_ids"],
            "attention_mask": (
                row["attention_mask"]
            ),
        }

        if "token_type_ids" in row:
            feature["token_type_ids"] = (
                row["token_type_ids"]
            )

        features.append(feature)

    padded = tokenizer.pad(
        features,
        padding=True,
        return_tensors="pt",
    )

    output = {
        "case_uid": [
            row["case_uid"]
            for row in batch
        ],
        "input_ids": padded["input_ids"],
        "attention_mask": (
            padded["attention_mask"]
        ),
    }

    if "token_type_ids" in padded:
        output["token_type_ids"] = (
            padded["token_type_ids"]
        )

    for target in TARGETS:
        output[target] = torch.tensor(
            [row[target] for row in batch],
            dtype=torch.long,
        )

    return output


@torch.no_grad()
def predict_condition(
    dataframe,
    text_column,
):
    inference_dataset = TextInferenceDataset(
        dataframe,
        text_column,
    )

    loader = DataLoader(
        inference_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda"),
        collate_fn=collate_batch,
    )

    case_uids = []
    true_values = {
        target: []
        for target in TARGETS
    }
    predictions = {
        target: []
        for target in TARGETS
    }

    for batch in loader:
        model_inputs = {
            "input_ids": batch[
                "input_ids"
            ].to(DEVICE),
            "attention_mask": batch[
                "attention_mask"
            ].to(DEVICE),
        }

        if "token_type_ids" in batch:
            model_inputs["token_type_ids"] = (
                batch["token_type_ids"].to(
                    DEVICE
                )
            )

        logits = model(**model_inputs)

        case_uids.extend(batch["case_uid"])

        for target in TARGETS:
            true_values[target].extend(
                batch[target]
                .cpu()
                .numpy()
                .tolist()
            )

            predictions[target].extend(
                logits[target]
                .argmax(dim=-1)
                .cpu()
                .numpy()
                .tolist()
            )

    return case_uids, true_values, predictions


# ------------------------------------------------------------
# 7. Verify full-test baseline reproduction
# ------------------------------------------------------------

print("\nVerifying KoELECTRA baseline...")

full_case_uids, full_true, full_pred = (
    predict_condition(
        full_df,
        "text_original",
    )
)


def calculate_metrics(
    true_values,
    predictions,
):
    target_metrics = {}
    macro_values = []

    for target in TARGETS:
        labels = list(
            range(NUM_LABELS[target])
        )

        accuracy = accuracy_score(
            true_values[target],
            predictions[target],
        )

        macro_f1 = f1_score(
            true_values[target],
            predictions[target],
            labels=labels,
            average="macro",
            zero_division=0,
        )

        target_metrics[target] = {
            "accuracy": float(accuracy),
            "macro_f1": float(macro_f1),
        }

        macro_values.append(macro_f1)

    true_matrix = np.column_stack([
        true_values[target]
        for target in TARGETS
    ])

    prediction_matrix = np.column_stack([
        predictions[target]
        for target in TARGETS
    ])

    return {
        "targets": target_metrics,
        "mean_macro_f1": float(
            np.mean(macro_values)
        ),
        "joint_exact_match": float(
            np.mean(
                np.all(
                    true_matrix
                    == prediction_matrix,
                    axis=1,
                )
            )
        ),
    }


full_metrics = calculate_metrics(
    full_true,
    full_pred,
)

print(
    " Reproduced Mean Macro-F1:",
    f"{full_metrics['mean_macro_f1']:.6f}",
)
print(
    " Reproduced Joint EM:",
    f"{full_metrics['joint_exact_match']:.6f}",
)

assert abs(
    full_metrics["mean_macro_f1"]
    - 0.400817
) < 0.001, "Baseline Macro-F1 reproduction failed"

assert abs(
    full_metrics["joint_exact_match"]
    - 0.509524
) < 0.001, "Baseline Joint EM reproduction failed"

print(" Baseline reproduction: PASSED")


# ------------------------------------------------------------
# 8. Run matched-subset ablation inference
# ------------------------------------------------------------

condition_outputs = {}

for condition in CONDITIONS:
    text_column = f"text_{condition}"

    print(f"\nRunning condition: {condition}")

    case_uids, true_values, predictions = (
        predict_condition(
            matched_df,
            text_column,
        )
    )

    condition_outputs[condition] = {
        "case_uids": case_uids,
        "true": true_values,
        "pred": predictions,
        "metrics": calculate_metrics(
            true_values,
            predictions,
        ),
    }


# ------------------------------------------------------------
# 9. Primary results and prediction flips
# ------------------------------------------------------------

original_output = condition_outputs["original"]

original_joint_predictions = np.column_stack([
    original_output["pred"][target]
    for target in TARGETS
])

primary_rows = []
target_rows = []
flip_rows = []

for condition in CONDITIONS:
    output = condition_outputs[condition]
    metrics = output["metrics"]

    condition_joint_predictions = (
        np.column_stack([
            output["pred"][target]
            for target in TARGETS
        ])
    )

    joint_flip_rate = float(
        np.mean(
            np.any(
                condition_joint_predictions
                != original_joint_predictions,
                axis=1,
            )
        )
    )

    primary_rows.append({
        "condition": condition,
        "support": len(matched_df),
        "mean_macro_f1": (
            metrics["mean_macro_f1"]
        ),
        "delta_mean_macro_f1": (
            metrics["mean_macro_f1"]
            - original_output["metrics"][
                "mean_macro_f1"
            ]
        ),
        "joint_exact_match": (
            metrics["joint_exact_match"]
        ),
        "delta_joint_exact_match": (
            metrics["joint_exact_match"]
            - original_output["metrics"][
                "joint_exact_match"
            ]
        ),
        "joint_flip_rate": joint_flip_rate,
    })

    for target in TARGETS:
        target_metrics = (
            metrics["targets"][target]
        )

        original_target_metrics = (
            original_output["metrics"][
                "targets"
            ][target]
        )

        target_rows.append({
            "condition": condition,
            "target": target,
            "support": len(matched_df),
            "accuracy": (
                target_metrics["accuracy"]
            ),
            "delta_accuracy": (
                target_metrics["accuracy"]
                - original_target_metrics[
                    "accuracy"
                ]
            ),
            "macro_f1": (
                target_metrics["macro_f1"]
            ),
            "delta_macro_f1": (
                target_metrics["macro_f1"]
                - original_target_metrics[
                    "macro_f1"
                ]
            ),
        })

        flip_rate = float(
            np.mean(
                np.asarray(
                    output["pred"][target]
                )
                != np.asarray(
                    original_output[
                        "pred"
                    ][target]
                )
            )
        )

        flip_rows.append({
            "condition": condition,
            "target": target,
            "support": len(matched_df),
            "prediction_flip_rate": (
                flip_rate
            ),
        })

primary_df = pd.DataFrame(primary_rows)
target_metrics_df = pd.DataFrame(target_rows)
flip_df = pd.DataFrame(flip_rows)


# ------------------------------------------------------------
# 10. Paired bootstrap
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)
bootstrap_rows = []

true_arrays = {
    target: np.asarray(
        original_output["true"][target]
    )
    for target in TARGETS
}

original_pred_arrays = {
    target: np.asarray(
        original_output["pred"][target]
    )
    for target in TARGETS
}


def indexed_metrics(
    true_arrays,
    pred_arrays,
    indices,
):
    macro_values = []

    for target in TARGETS:
        macro_values.append(
            f1_score(
                true_arrays[target][indices],
                pred_arrays[target][indices],
                labels=list(
                    range(NUM_LABELS[target])
                ),
                average="macro",
                zero_division=0,
            )
        )

    true_matrix = np.column_stack([
        true_arrays[target][indices]
        for target in TARGETS
    ])

    prediction_matrix = np.column_stack([
        pred_arrays[target][indices]
        for target in TARGETS
    ])

    return (
        float(np.mean(macro_values)),
        float(
            np.mean(
                np.all(
                    true_matrix
                    == prediction_matrix,
                    axis=1,
                )
            )
        ),
    )


for condition in [
    "marker_removed",
    "sentence_removed",
]:
    condition_pred_arrays = {
        target: np.asarray(
            condition_outputs[condition][
                "pred"
            ][target]
        )
        for target in TARGETS
    }

    for iteration in range(
        BOOTSTRAP_ITERATIONS
    ):
        indices = rng.integers(
            0,
            len(matched_df),
            size=len(matched_df),
        )

        original_f1, original_em = (
            indexed_metrics(
                true_arrays,
                original_pred_arrays,
                indices,
            )
        )

        condition_f1, condition_em = (
            indexed_metrics(
                true_arrays,
                condition_pred_arrays,
                indices,
            )
        )

        bootstrap_rows.extend([
            {
                "condition": condition,
                "iteration": iteration,
                "metric": "mean_macro_f1",
                "delta": (
                    condition_f1
                    - original_f1
                ),
            },
            {
                "condition": condition,
                "iteration": iteration,
                "metric": (
                    "joint_exact_match"
                ),
                "delta": (
                    condition_em
                    - original_em
                ),
            },
        ])

bootstrap_df = pd.DataFrame(bootstrap_rows)

bootstrap_summary_rows = []

for (
    condition,
    metric,
), group in bootstrap_df.groupby(
    ["condition", "metric"]
):
    values = group["delta"].to_numpy()

    lower = float(
        np.percentile(values, 2.5)
    )
    upper = float(
        np.percentile(values, 97.5)
    )

    observed_row = primary_df.loc[
        primary_df["condition"].eq(condition)
    ].iloc[0]

    observed_column = (
        "delta_mean_macro_f1"
        if metric == "mean_macro_f1"
        else "delta_joint_exact_match"
    )

    bootstrap_summary_rows.append({
        "condition": condition,
        "metric": metric,
        "observed_delta": float(
            observed_row[observed_column]
        ),
        "bootstrap_mean_delta": float(
            values.mean()
        ),
        "ci_lower_95": lower,
        "ci_upper_95": upper,
        "ci_excludes_zero": bool(
            lower > 0 or upper < 0
        ),
        "iterations": (
            BOOTSTRAP_ITERATIONS
        ),
    })

bootstrap_summary_df = pd.DataFrame(
    bootstrap_summary_rows
)


# ------------------------------------------------------------
# 11. Save private predictions and aggregate outputs
# ------------------------------------------------------------

prediction_df = matched_df[
    ["case_uid", *TARGETS]
].copy()

for condition in CONDITIONS:
    for target in TARGETS:
        prediction_df[
            f"pred_{condition}_{target}"
        ] = condition_outputs[condition][
            "pred"
        ][target]

private_prediction_path = (
    PREDICTION_DIR
    / "koelectra_legal_conclusion_predictions_private.parquet"
)

prediction_df.to_parquet(
    private_prediction_path,
    index=False,
)

primary_path = (
    METRICS_DIR / "primary_results.csv"
)

target_metrics_path = (
    METRICS_DIR / "target_metrics.csv"
)

flip_path = (
    METRICS_DIR / "prediction_flip_summary.csv"
)

bootstrap_records_path = (
    BOOTSTRAP_DIR / "bootstrap_records.csv"
)

bootstrap_summary_path = (
    BOOTSTRAP_DIR / "bootstrap_summary.csv"
)

primary_df.to_csv(
    primary_path,
    index=False,
)

target_metrics_df.to_csv(
    target_metrics_path,
    index=False,
)

flip_df.to_csv(
    flip_path,
    index=False,
)

bootstrap_df.to_csv(
    bootstrap_records_path,
    index=False,
)

bootstrap_summary_df.to_csv(
    bootstrap_summary_path,
    index=False,
)

summary = {
    "experiment_name": EXPERIMENT_NAME,
    "status": "completed",
    "model_name": MODEL_NAME,
    "model_family": "ELECTRA",
    "seed": SEED,
    "total_test_cases": len(full_df),
    "matched_cases": len(matched_df),
    "matched_rate": (
        len(matched_df) / len(full_df)
    ),
    "conditions": CONDITIONS,
    "baseline_reproduction": {
        "mean_macro_f1": (
            full_metrics["mean_macro_f1"]
        ),
        "joint_exact_match": (
            full_metrics[
                "joint_exact_match"
            ]
        ),
    },
    "primary_results": (
        primary_df.to_dict(
            orient="records"
        )
    ),
    "bootstrap_summary": (
        bootstrap_summary_df.to_dict(
            orient="records"
        )
    ),
    "privacy": {
        "transformed_text_private": True,
        "per_case_predictions_private": True,
        "aggregate_metrics_publishable": True,
    },
}

summary_path = (
    RESULT_ROOT / "experiment_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 12. Final output
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PRIMARY RESULTS")
print("=" * 72)

print(
    primary_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\n" + "=" * 72)
print("TARGET METRICS")
print("=" * 72)

print(
    target_metrics_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\n" + "=" * 72)
print("PREDICTION FLIPS")
print("=" * 72)

print(
    flip_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\n" + "=" * 72)
print("PAIRED BOOTSTRAP SUMMARY")
print("=" * 72)

print(
    bootstrap_summary_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nSaved:")
print(" Private transformed data:", private_data_path)
print(" Private predictions:", private_prediction_path)
print(" Primary metrics:", primary_path)
print(" Target metrics:", target_metrics_path)
print(" Flip summary:", flip_path)
print(" Bootstrap summary:", bootstrap_summary_path)
print(" Experiment summary:", summary_path)

print("\nKoELECTRA LEGAL-CONCLUSION ABLATION: COMPLETED")


## Execution block 2

In [ ]:
import os
# ============================================================
# KoELECTRA OUTCOME-CUE INSERTION GENERALIZATION
# Continuation cell — uses the currently loaded KoELECTRA model
# ============================================================

import gc
import json
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# 1. Verify previous runtime
# ------------------------------------------------------------

required_objects = [
    "PROJECT_ROOT",
    "model",
    "tokenizer",
    "test_raw",
    "TARGETS",
    "NUM_LABELS",
    "DEVICE",
    "predict_condition",
    "calculate_metrics",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    "Previous KoELECTRA runtime objects are missing: "
    f"{missing_objects}. Do not restart the runtime."
)

EXPERIMENT_NAME = (
    "koelectra_outcome_cue_insertion_v1"
)

RESULT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "audit_experiments"
    / EXPERIMENT_NAME
)

PRIVATE_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / EXPERIMENT_NAME
)

PREDICTION_DIR = RESULT_ROOT / "predictions"
METRICS_DIR = RESULT_ROOT / "metrics"
BOOTSTRAP_DIR = RESULT_ROOT / "bootstrap"

for path in [
    RESULT_ROOT,
    PRIVATE_DATA_DIR,
    PREDICTION_DIR,
    METRICS_DIR,
    BOOTSTRAP_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

CONDITIONS = [
    "original",
    "neutral",
    "lenient",
    "severe",
]

CUES = {
    "neutral": (
        "이 사건의 구체적인 사실관계는 "
        "위에서 서술한 바와 같다."
    ),
    "lenient": (
        "이 사건은 참작할 사정이 충분하여 "
        "관대한 처벌이 타당하다."
    ),
    "severe": (
        "이 사건은 죄책이 무거워 "
        "엄한 처벌이 불가피하다."
    ),
}

MAX_LENGTH = 512
BOOTSTRAP_ITERATIONS = 1000
SEED = 42

print("=" * 72)
print("KoELECTRA OUTCOME-CUE INSERTION")
print("=" * 72)
print("Device:", DEVICE)
print("Result root:", RESULT_ROOT)


# ------------------------------------------------------------
# 2. Build exact cue variants
# ------------------------------------------------------------

def normalize_korean(text):
    return unicodedata.normalize(
        "NFC",
        str(text).strip(),
    )


def append_outcome_cue(
    text,
    condition,
):
    text = normalize_korean(text)
    condition = condition.lower()

    if condition == "original":
        return text

    assert condition in CUES

    return normalize_korean(
        f"{text} {CUES[condition]}"
    )


records = []

for row in test_raw:
    record = {
        "case_uid": str(row["id"]),
    }

    original_text = normalize_korean(
        row["facts"]
    )

    for condition in CONDITIONS:
        record[f"text_{condition}"] = (
            append_outcome_cue(
                original_text,
                condition,
            )
        )

    for target in TARGETS:
        record[target] = int(
            row["label"][target]
        )

    records.append(record)

cue_df = pd.DataFrame(records)

assert len(cue_df) == 1050
assert cue_df["case_uid"].nunique() == 1050
assert cue_df["case_uid"].notna().all()

for condition in [
    "neutral",
    "lenient",
    "severe",
]:
    assert (
        cue_df[f"text_{condition}"]
        != cue_df["text_original"]
    ).all()

print("Cases:", len(cue_df))
print("Cue construction: PASSED")


# ------------------------------------------------------------
# 3. Token visibility
# ------------------------------------------------------------

def get_token_lengths(texts, batch_size=128):
    lengths = []

    for start in range(
        0,
        len(texts),
        batch_size,
    ):
        batch_texts = texts[
            start:start + batch_size
        ]

        encoded = tokenizer(
            batch_texts,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_length=True,
        )

        if "length" in encoded:
            lengths.extend(
                [
                    int(value)
                    for value in encoded["length"]
                ]
            )
        else:
            lengths.extend(
                [
                    len(input_ids)
                    for input_ids
                    in encoded["input_ids"]
                ]
            )

    return np.asarray(
        lengths,
        dtype=np.int64,
    )


for condition in CONDITIONS:
    cue_df[
        f"token_length_{condition}"
    ] = get_token_lengths(
        cue_df[f"text_{condition}"].tolist()
    )

cue_df["all_cues_fully_visible"] = True

for condition in CONDITIONS:
    cue_df["all_cues_fully_visible"] &= (
        cue_df[
            f"token_length_{condition}"
        ] <= MAX_LENGTH
    )

visible_mask = cue_df[
    "all_cues_fully_visible"
].to_numpy(dtype=bool)

visible_count = int(visible_mask.sum())
visible_rate = visible_count / len(cue_df)

print("\nCue visibility:")
print(" Total cases:", len(cue_df))
print(" Fully visible:", visible_count)
print(" Fully visible rate:", f"{visible_rate:.2%}")

for condition in CONDITIONS:
    lengths = cue_df[
        f"token_length_{condition}"
    ]

    print(
        f" {condition:10s} | "
        f"mean={lengths.mean():.2f} | "
        f"max={lengths.max()} | "
        f">512={(lengths > MAX_LENGTH).sum()}"
    )

assert visible_count > 0


# ------------------------------------------------------------
# 4. Save private transformed data
# ------------------------------------------------------------

private_data_path = (
    PRIVATE_DATA_DIR
    / "koelectra_outcome_cue_test_private.parquet"
)

cue_df.to_parquet(
    private_data_path,
    index=False,
)


# ------------------------------------------------------------
# 5. Run inference for all four conditions
# ------------------------------------------------------------

condition_outputs = {}

for condition in CONDITIONS:
    print(f"\nRunning condition: {condition}")

    case_uids, true_values, predictions = (
        predict_condition(
            cue_df,
            f"text_{condition}",
        )
    )

    assert len(case_uids) == 1050

    condition_outputs[condition] = {
        "case_uids": case_uids,
        "true": {
            target: np.asarray(
                true_values[target],
                dtype=np.int64,
            )
            for target in TARGETS
        },
        "pred": {
            target: np.asarray(
                predictions[target],
                dtype=np.int64,
            )
            for target in TARGETS
        },
    }


# ------------------------------------------------------------
# 6. Full-test baseline reproduction
# ------------------------------------------------------------

full_original_metrics = calculate_metrics(
    condition_outputs["original"]["true"],
    condition_outputs["original"]["pred"],
)

print("\nBaseline reproduction:")
print(
    " Mean Macro-F1:",
    f"{full_original_metrics['mean_macro_f1']:.6f}",
)
print(
    " Joint EM:",
    f"{full_original_metrics['joint_exact_match']:.6f}",
)

assert abs(
    full_original_metrics["mean_macro_f1"]
    - 0.400817
) < 0.001

assert abs(
    full_original_metrics["joint_exact_match"]
    - 0.509524
) < 0.001

print(" Baseline reproduction: PASSED")


# ------------------------------------------------------------
# 7. Metrics helper for selected indices
# ------------------------------------------------------------

def metrics_on_mask(
    true_arrays,
    prediction_arrays,
    mask,
):
    target_metrics = {}
    macro_f1_values = []

    for target in TARGETS:
        y_true = true_arrays[target][mask]
        y_pred = prediction_arrays[target][mask]

        labels = list(
            range(NUM_LABELS[target])
        )

        accuracy = accuracy_score(
            y_true,
            y_pred,
        )

        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0,
        )

        weighted_f1 = f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="weighted",
            zero_division=0,
        )

        target_metrics[target] = {
            "accuracy": float(accuracy),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(
                weighted_f1
            ),
        }

        macro_f1_values.append(macro_f1)

    true_matrix = np.column_stack([
        true_arrays[target][mask]
        for target in TARGETS
    ])

    prediction_matrix = np.column_stack([
        prediction_arrays[target][mask]
        for target in TARGETS
    ])

    return {
        "targets": target_metrics,
        "mean_macro_f1": float(
            np.mean(macro_f1_values)
        ),
        "joint_exact_match": float(
            np.mean(
                np.all(
                    true_matrix
                    == prediction_matrix,
                    axis=1,
                )
            )
        ),
    }


visible_metrics = {}

for condition in CONDITIONS:
    visible_metrics[condition] = (
        metrics_on_mask(
            condition_outputs[condition][
                "true"
            ],
            condition_outputs[condition][
                "pred"
            ],
            visible_mask,
        )
    )


# ------------------------------------------------------------
# 8. Primary results and flips
# ------------------------------------------------------------

original_metrics = visible_metrics["original"]

original_joint_predictions = (
    np.column_stack([
        condition_outputs["original"][
            "pred"
        ][target][visible_mask]
        for target in TARGETS
    ])
)

primary_rows = []
target_rows = []
flip_rows = []

for condition in CONDITIONS:
    metrics = visible_metrics[condition]

    condition_joint_predictions = (
        np.column_stack([
            condition_outputs[condition][
                "pred"
            ][target][visible_mask]
            for target in TARGETS
        ])
    )

    joint_flip_rate = float(
        np.mean(
            np.any(
                condition_joint_predictions
                != original_joint_predictions,
                axis=1,
            )
        )
    )

    primary_rows.append({
        "condition": condition,
        "support": visible_count,
        "mean_macro_f1": (
            metrics["mean_macro_f1"]
        ),
        "delta_mean_macro_f1": (
            metrics["mean_macro_f1"]
            - original_metrics[
                "mean_macro_f1"
            ]
        ),
        "joint_exact_match": (
            metrics["joint_exact_match"]
        ),
        "delta_joint_exact_match": (
            metrics["joint_exact_match"]
            - original_metrics[
                "joint_exact_match"
            ]
        ),
        "joint_flip_rate": (
            joint_flip_rate
        ),
    })

    for target in TARGETS:
        condition_target = (
            metrics["targets"][target]
        )

        original_target = (
            original_metrics["targets"][
                target
            ]
        )

        prediction_flip_rate = float(
            np.mean(
                condition_outputs[condition][
                    "pred"
                ][target][visible_mask]
                != condition_outputs[
                    "original"
                ]["pred"][target][visible_mask]
            )
        )

        target_rows.append({
            "condition": condition,
            "target": target,
            "support": visible_count,
            "accuracy": (
                condition_target["accuracy"]
            ),
            "delta_accuracy": (
                condition_target["accuracy"]
                - original_target["accuracy"]
            ),
            "macro_f1": (
                condition_target["macro_f1"]
            ),
            "delta_macro_f1": (
                condition_target["macro_f1"]
                - original_target[
                    "macro_f1"
                ]
            ),
        })

        flip_rows.append({
            "condition": condition,
            "target": target,
            "support": visible_count,
            "prediction_flip_rate": (
                prediction_flip_rate
            ),
        })

primary_df = pd.DataFrame(primary_rows)
target_metrics_df = pd.DataFrame(
    target_rows
)
flip_df = pd.DataFrame(flip_rows)


# ------------------------------------------------------------
# 9. Paired bootstrap
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)

visible_indices = np.flatnonzero(
    visible_mask
)

bootstrap_rows = []


def bootstrap_metrics(
    true_arrays,
    prediction_arrays,
    sampled_global_indices,
):
    macro_values = []

    for target in TARGETS:
        y_true = true_arrays[target][
            sampled_global_indices
        ]

        y_pred = prediction_arrays[target][
            sampled_global_indices
        ]

        macro_values.append(
            f1_score(
                y_true,
                y_pred,
                labels=list(
                    range(NUM_LABELS[target])
                ),
                average="macro",
                zero_division=0,
            )
        )

    true_matrix = np.column_stack([
        true_arrays[target][
            sampled_global_indices
        ]
        for target in TARGETS
    ])

    prediction_matrix = np.column_stack([
        prediction_arrays[target][
            sampled_global_indices
        ]
        for target in TARGETS
    ])

    return (
        float(np.mean(macro_values)),
        float(
            np.mean(
                np.all(
                    true_matrix
                    == prediction_matrix,
                    axis=1,
                )
            )
        ),
    )


for condition in [
    "neutral",
    "lenient",
    "severe",
]:
    for iteration in range(
        BOOTSTRAP_ITERATIONS
    ):
        sampled_positions = rng.integers(
            0,
            visible_count,
            size=visible_count,
        )

        sampled_indices = visible_indices[
            sampled_positions
        ]

        original_f1, original_em = (
            bootstrap_metrics(
                condition_outputs["original"][
                    "true"
                ],
                condition_outputs["original"][
                    "pred"
                ],
                sampled_indices,
            )
        )

        condition_f1, condition_em = (
            bootstrap_metrics(
                condition_outputs[condition][
                    "true"
                ],
                condition_outputs[condition][
                    "pred"
                ],
                sampled_indices,
            )
        )

        bootstrap_rows.extend([
            {
                "condition": condition,
                "iteration": iteration,
                "metric": "mean_macro_f1",
                "delta": (
                    condition_f1
                    - original_f1
                ),
            },
            {
                "condition": condition,
                "iteration": iteration,
                "metric": (
                    "joint_exact_match"
                ),
                "delta": (
                    condition_em
                    - original_em
                ),
            },
        ])

bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

bootstrap_summary_rows = []

for (
    condition,
    metric,
), group in bootstrap_df.groupby(
    ["condition", "metric"]
):
    values = group["delta"].to_numpy()

    lower = float(
        np.percentile(values, 2.5)
    )
    upper = float(
        np.percentile(values, 97.5)
    )

    observed_row = primary_df.loc[
        primary_df["condition"].eq(
            condition
        )
    ].iloc[0]

    delta_column = (
        "delta_mean_macro_f1"
        if metric == "mean_macro_f1"
        else "delta_joint_exact_match"
    )

    bootstrap_summary_rows.append({
        "condition": condition,
        "metric": metric,
        "observed_delta": float(
            observed_row[delta_column]
        ),
        "bootstrap_mean_delta": float(
            values.mean()
        ),
        "ci_lower_95": lower,
        "ci_upper_95": upper,
        "ci_excludes_zero": bool(
            lower > 0 or upper < 0
        ),
        "iterations": (
            BOOTSTRAP_ITERATIONS
        ),
    })

bootstrap_summary_df = pd.DataFrame(
    bootstrap_summary_rows
)


# ------------------------------------------------------------
# 10. Save predictions and metrics
# ------------------------------------------------------------

prediction_df = cue_df[
    [
        "case_uid",
        "all_cues_fully_visible",
        *TARGETS,
    ]
].copy()

for condition in CONDITIONS:
    for target in TARGETS:
        prediction_df[
            f"pred_{condition}_{target}"
        ] = condition_outputs[condition][
            "pred"
        ][target]

private_prediction_path = (
    PREDICTION_DIR
    / "koelectra_outcome_cue_predictions_private.parquet"
)

prediction_df.to_parquet(
    private_prediction_path,
    index=False,
)

primary_path = (
    METRICS_DIR / "primary_results.csv"
)

target_metrics_path = (
    METRICS_DIR / "target_metrics.csv"
)

flip_path = (
    METRICS_DIR / "prediction_flip_summary.csv"
)

bootstrap_records_path = (
    BOOTSTRAP_DIR
    / "bootstrap_records.csv"
)

bootstrap_summary_path = (
    BOOTSTRAP_DIR
    / "bootstrap_summary.csv"
)

primary_df.to_csv(
    primary_path,
    index=False,
)

target_metrics_df.to_csv(
    target_metrics_path,
    index=False,
)

flip_df.to_csv(
    flip_path,
    index=False,
)

bootstrap_df.to_csv(
    bootstrap_records_path,
    index=False,
)

bootstrap_summary_df.to_csv(
    bootstrap_summary_path,
    index=False,
)

summary = {
    "experiment_name": EXPERIMENT_NAME,
    "status": "completed",
    "model_name": (
        "monologg/koelectra-base-v3-discriminator"
    ),
    "model_family": "ELECTRA",
    "seed": SEED,
    "total_cases": len(cue_df),
    "fully_visible_cases": visible_count,
    "fully_visible_rate": visible_rate,
    "conditions": CONDITIONS,
    "cues": CUES,
    "baseline_reproduction": {
        "mean_macro_f1": (
            full_original_metrics[
                "mean_macro_f1"
            ]
        ),
        "joint_exact_match": (
            full_original_metrics[
                "joint_exact_match"
            ]
        ),
    },
    "primary_results": (
        primary_df.to_dict(
            orient="records"
        )
    ),
    "bootstrap_summary": (
        bootstrap_summary_df.to_dict(
            orient="records"
        )
    ),
    "privacy": {
        "transformed_text_private": True,
        "per_case_predictions_private": True,
        "aggregate_metrics_publishable": True,
    },
}

summary_path = (
    RESULT_ROOT
    / "experiment_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 11. Final output
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PRIMARY RESULTS")
print("=" * 72)

print(
    primary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\n" + "=" * 72)
print("TARGET RESULTS")
print("=" * 72)

print(
    target_metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\n" + "=" * 72)
print("PREDICTION FLIPS")
print("=" * 72)

print(
    flip_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\n" + "=" * 72)
print("PAIRED BOOTSTRAP SUMMARY")
print("=" * 72)

print(
    bootstrap_summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nSaved:")
print(" Private data:", private_data_path)
print(
    " Private predictions:",
    private_prediction_path,
)
print(" Primary metrics:", primary_path)
print(
    " Bootstrap summary:",
    bootstrap_summary_path,
)
print(" Summary:", summary_path)

print("\nKoELECTRA OUTCOME-CUE INSERTION: COMPLETED")
